In [9]:
from torchvision import datasets, transforms as T, models
import numpy as np
from torchinfo import summary
import torch.nn as nn
import torch
from torch.utils.data import DataLoader
import torch.optim as optim

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [11]:
transform_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.5071, 0.4867, 0.4408], [0.2675, 0.2565, 0.2761])
])
transform_test = T.Compose([
    T.ToTensor(),
    T.Normalize([0.5071, 0.4867, 0.4408], [0.2675, 0.2565, 0.2761])
])
train_ds = datasets.CIFAR100("./data/", train=True, transform=transform_train, download=True)
test_ds = datasets.CIFAR100("./data/", train=False, transform=transform_test, download=True)
print(train_ds.data.shape, test_ds.data.shape)
print(len(np.unique(test_ds.classes)))

(50000, 32, 32, 3) (10000, 32, 32, 3)
100


In [12]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for param in model.parameters():
    param.requires_grad = False
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Linear(model.fc.in_features, 100)
model = model.to(device)
summary(model)

Layer (type:depth-idx)                   Param #
ResNet                                   --
├─Conv2d: 1-1                            1,728
├─BatchNorm2d: 1-2                       (128)
├─ReLU: 1-3                              --
├─Identity: 1-4                          --
├─Sequential: 1-5                        --
│    └─BasicBlock: 2-1                   --
│    │    └─Conv2d: 3-1                  (36,864)
│    │    └─BatchNorm2d: 3-2             (128)
│    │    └─ReLU: 3-3                    --
│    │    └─Conv2d: 3-4                  (36,864)
│    │    └─BatchNorm2d: 3-5             (128)
│    └─BasicBlock: 2-2                   --
│    │    └─Conv2d: 3-6                  (36,864)
│    │    └─BatchNorm2d: 3-7             (128)
│    │    └─ReLU: 3-8                    --
│    │    └─Conv2d: 3-9                  (36,864)
│    │    └─BatchNorm2d: 3-10            (128)
├─Sequential: 1-6                        --
│    └─BasicBlock: 2-3                   --
│    │    └─Conv2d: 3-11     

In [13]:
import copy

def train(model, epochs, optimizer, train_loader, test_loader, criterion, device):
    best_accuracy = 0.0
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_X, batch_y in train_loader:
            # batch_X = batch_X.reshape(batch_X.size(0), -1)
            batch_X, batch_y = batch_X.to(device, non_blocking=True), batch_y.to(device, non_blocking=True)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        train_loss = running_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0   
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device, non_blocking=True)
                batch_y = batch_y.to(device, non_blocking=True)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
        scheduler.step()
        val_loss = val_loss / len(test_loader) # Replace with len(val_loader)
        val_accuracy = correct / total
        
        # --- Save Best Model ---
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            best_model_wts = copy.deepcopy(model.state_dict())
    
        print(f"Epoch: {epoch+1}/{epochs}\tTrain Loss: {train_loss:.4f}\tVal Loss: {val_loss:.4f}\tVal Acc: {val_accuracy:.4f}")
    model.load_state_dict(best_model_wts)
    print(f"Training complete. Best Validation Accuracy: {best_accuracy:.4f}")

In [14]:
batch_size = 128
epochs = 30

In [15]:
train_loader = DataLoader(train_ds, batch_size, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(test_ds, batch_size, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

In [16]:
train(model, epochs, optimizer, train_loader, test_loader, criterion, device)

Epoch: 1/30	Train Loss: 3.6855	Val Loss: 3.0818	Val Acc: 0.2571
Epoch: 2/30	Train Loss: 2.8481	Val Loss: 2.6489	Val Acc: 0.3442
Epoch: 3/30	Train Loss: 2.4942	Val Loss: 2.3813	Val Acc: 0.3924
Epoch: 4/30	Train Loss: 2.2787	Val Loss: 2.2461	Val Acc: 0.4218
Epoch: 5/30	Train Loss: 2.1435	Val Loss: 2.1071	Val Acc: 0.4519
Epoch: 6/30	Train Loss: 2.0423	Val Loss: 2.0765	Val Acc: 0.4578
Epoch: 7/30	Train Loss: 1.9860	Val Loss: 1.9597	Val Acc: 0.4863
Epoch: 8/30	Train Loss: 1.9168	Val Loss: 1.9460	Val Acc: 0.4840
Epoch: 9/30	Train Loss: 1.8818	Val Loss: 1.9018	Val Acc: 0.4955
Epoch: 10/30	Train Loss: 1.8416	Val Loss: 1.8632	Val Acc: 0.5015
Epoch: 11/30	Train Loss: 1.8125	Val Loss: 1.8341	Val Acc: 0.5105
Epoch: 12/30	Train Loss: 1.7795	Val Loss: 1.8529	Val Acc: 0.5069
Epoch: 13/30	Train Loss: 1.7501	Val Loss: 1.8361	Val Acc: 0.5160
Epoch: 14/30	Train Loss: 1.7293	Val Loss: 1.8004	Val Acc: 0.5204
Epoch: 15/30	Train Loss: 1.7133	Val Loss: 1.7788	Val Acc: 0.5246
Epoch: 16/30	Train Loss: 1.7000	Va

In [17]:
for param in model.parameters():
    param.requires_grad = True

In [18]:
train(model, epochs, optimizer, train_loader, test_loader, criterion, device)

Epoch: 1/30	Train Loss: 1.5653	Val Loss: 1.6778	Val Acc: 0.5463
Epoch: 2/30	Train Loss: 1.5697	Val Loss: 1.6701	Val Acc: 0.5476
Epoch: 3/30	Train Loss: 1.5670	Val Loss: 1.6682	Val Acc: 0.5488
Epoch: 4/30	Train Loss: 1.5711	Val Loss: 1.6659	Val Acc: 0.5513
Epoch: 5/30	Train Loss: 1.5739	Val Loss: 1.6758	Val Acc: 0.5472
Epoch: 6/30	Train Loss: 1.5780	Val Loss: 1.6853	Val Acc: 0.5420
Epoch: 7/30	Train Loss: 1.5761	Val Loss: 1.6720	Val Acc: 0.5467
Epoch: 8/30	Train Loss: 1.5769	Val Loss: 1.6853	Val Acc: 0.5449
Epoch: 9/30	Train Loss: 1.5800	Val Loss: 1.6995	Val Acc: 0.5413
Epoch: 10/30	Train Loss: 1.5838	Val Loss: 1.7174	Val Acc: 0.5344
Epoch: 11/30	Train Loss: 1.5890	Val Loss: 1.6750	Val Acc: 0.5444
Epoch: 12/30	Train Loss: 1.6018	Val Loss: 1.6697	Val Acc: 0.5463
Epoch: 13/30	Train Loss: 1.5939	Val Loss: 1.7136	Val Acc: 0.5382
Epoch: 14/30	Train Loss: 1.6024	Val Loss: 1.6879	Val Acc: 0.5473
Epoch: 15/30	Train Loss: 1.5955	Val Loss: 1.6784	Val Acc: 0.5478
Epoch: 16/30	Train Loss: 1.6076	Va